# End-to-end BERT package recommender

In [ ]:
# Run once if needed:
# !pip install transformers iterative-stratification sqlalchemy psycopg2-binary

import copy
import os
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dotenv import load_dotenv
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

SEED = 42
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
MAX_LENGTH = 256
BATCH_SIZE = 8
TEST_SIZE = 0.2
MIN_PACKAGE_COUNT = 5
EPOCHS = 20
PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.5
MIN_DELTA = 1e-4
BERT_LEARNING_RATE = 2e-5
HEAD_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Load data

In [ ]:
from dotenv import load_dotenv
import os

from markdown_it.rules_block import list_block
from rich.diagnose import report

# from cosine_similarity import valid_deps

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')
curr_dir = os.getcwd()
postgres_env_path = os.path.join(curr_dir, "docker-compose\\postgres_db", ".env")
load_dotenv(dotenv_path=postgres_env_path)
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB = os.getenv("POSTGRES_DB")
DB_PORT = os.getenv("DB_PORT")

In [ ]:
# Choose one source. The parquet file must contain: id, fullname/name, desc, deps.
USE_PARQUET = False
PARQUET_PATH = '/content/shared/mnt/db.parquet'

if USE_PARQUET:
    df = pd.read_parquet(PARQUET_PATH)
else:
    from sqlalchemy import create_engine

    load_dotenv()
    db_url = (
        f"postgresql://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
        f"@localhost:{os.getenv('DB_PORT')}/{os.getenv('POSTGRES_DB')}"
    )
    engine = create_engine(db_url)
    df = pd.read_sql(
        '''
        SELECT id, name AS fullname, "desc", deps
        FROM repo
        WHERE cardinality(deps) > 3
          AND "desc" IS NOT NULL
        ORDER BY id
        ''',
        con=engine,
    )

if 'fullname' not in df.columns and 'name' in df.columns:
    df = df.rename(columns={'name': 'fullname'})

required_columns = {'desc', 'deps', 'fullname'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing columns: {sorted(missing_columns)}')

print(df[['fullname', 'desc', 'deps']].head())
print('Rows:', len(df))

## Multilabel-stratified 80/20 split

In [ ]:
split_df = df[['desc', 'deps', 'fullname']].copy()
split_df = split_df[split_df['desc'].notna() & split_df['deps'].notna()].reset_index(drop=True)

split_df['desc'] = split_df['desc'].astype(str)
split_df = split_df[split_df['desc'].str.strip().str.len() > 0].reset_index(drop=True)

unique_deps = Counter(
    package
    for deps in split_df['deps']
    for package in set(deps)
)

valid_deps = {
    package for package, count in unique_deps.items()
    if count >= MIN_PACKAGE_COUNT
}

split_df['deps'] = split_df['deps'].apply(
    lambda deps: sorted(set(deps).intersection(valid_deps))
)
split_df = split_df[split_df['deps'].map(len) > 0].reset_index(drop=True)

split_mlb = MultiLabelBinarizer(classes=sorted(valid_deps))
all_labels = split_mlb.fit_transform(split_df['deps'])
splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=TEST_SIZE, random_state=SEED
)
train_indices, test_indices = next(
    splitter.split(np.zeros(len(split_df)), all_labels)
)
train_df = split_df.iloc[train_indices].copy().reset_index(drop=True)
test_df = split_df.iloc[test_indices].copy().reset_index(drop=True)

train_packages = set().union(*(set(deps) for deps in train_df['deps']))
test_packages = set().union(*(set(deps) for deps in test_df['deps']))
shared_packages = train_packages.intersection(test_packages)

for frame in (train_df, test_df):
    frame['deps'] = frame['deps'].apply(
        lambda deps: sorted(set(deps).intersection(shared_packages))
    )
train_df = train_df[train_df['deps'].map(len) > 0].reset_index(drop=True)
test_df = test_df[test_df['deps'].map(len) > 0].reset_index(drop=True)

mlb = MultiLabelBinarizer(classes=sorted(shared_packages))
y_train_encoded = mlb.fit_transform(train_df['deps']).astype(np.float32)
y_test_encoded = mlb.transform(test_df['deps']).astype(np.float32)
num_classes = len(mlb.classes_)

print(f'Train: {len(train_df)} ({len(train_df) / (len(train_df) + len(test_df)):.2%})')
print(f'Test:  {len(test_df)} ({len(test_df) / (len(train_df) + len(test_df)):.2%})')
print('Classes:', num_classes)
print('Train-only packages removed:', len(train_packages - test_packages))
print('Test-only packages removed:', len(test_packages - train_packages))
assert not np.any(y_train_encoded.sum(axis=1) == 0)
assert not np.any(y_test_encoded.sum(axis=1) == 0)

## Tokenizer, Dataset and DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # BERT tokenizer model

class DescriptionDataset(Dataset):
    def __init__(self, descriptions, labels, tokenizer, max_length):
        self.descriptions = list(descriptions)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.descriptions[index],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': self.labels[index],
        }

train_dataset = DescriptionDataset(
    train_df['desc'], y_train_encoded, tokenizer, MAX_LENGTH
)
test_dataset = DescriptionDataset(
    test_df['desc'], y_test_encoded, tokenizer, MAX_LENGTH
)
train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available()
)
test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available()
)
print('Train batches:', len(train_dataloader))
print('Test batches:', len(test_dataloader))

## Full BERT fine-tuning with an MLP head on CLS

In [ ]:
class BertPackageClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding)

bert_model = BertPackageClassifier(MODEL_NAME, num_classes).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    [
        {'params': bert_model.bert.parameters(), 'lr': BERT_LEARNING_RATE},
        {'params': bert_model.classifier.parameters(), 'lr': HEAD_LEARNING_RATE},
    ],
    weight_decay=WEIGHT_DECAY,
)
print('Trainable parameters:', sum(p.numel() for p in bert_model.parameters() if p.requires_grad))

In [ ]:
best_test_f1 = -1.0
best_epoch = 0
epochs_no_improve = 0
best_model_wts = copy.deepcopy(bert_model.state_dict())

for epoch in range(EPOCHS):
    bert_model.train()
    total_train_loss = 0.0

    for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1} train'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = bert_model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)
    bert_model.eval()
    total_test_loss = 0.0
    true_batches = []
    pred_batches = []

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc=f'Epoch {epoch + 1} test'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            logits = bert_model(input_ids, attention_mask)
            total_test_loss += criterion(logits, labels).item()
            predictions = (torch.sigmoid(logits) >= EARLY_STOPPING_THRESHOLD).int()
            true_batches.append(labels.cpu().numpy().astype(np.int8))
            pred_batches.append(predictions.cpu().numpy().astype(np.int8))

    test_true = np.concatenate(true_batches, axis=0)
    test_pred = np.concatenate(pred_batches, axis=0)
    test_f1 = f1_score(test_true, test_pred, average='samples', zero_division=0)
    avg_test_loss = total_test_loss / len(test_dataloader)
    print(
        f'Epoch [{epoch + 1}/{EPOCHS}] | Train loss: {avg_train_loss:.4f} | '
        f'Test loss: {avg_test_loss:.4f} | Samples F1@0.50: {test_f1:.4f}'
    )

    if test_f1 > best_test_f1 + MIN_DELTA:
        best_test_f1 = test_f1
        best_epoch = epoch + 1
        epochs_no_improve = 0
        best_model_wts = copy.deepcopy(bert_model.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping after {PATIENCE} epochs without F1 improvement.')
            break

bert_model.load_state_dict(best_model_wts)
print(f'Restored epoch {best_epoch}; best samples F1: {best_test_f1:.4f}')

## Precision@k, Recall@k and F1@k

In [ ]:
def evaluate_at_k(
    model, dataloader, k_values=(1, 2, 3, 5, 9, 10, 11, 15, 20), device=device
):
    model.eval()
    precisions = {k: [] for k in k_values}
    recalls = {k: [] for k in k_values}
    f1_scores = {k: [] for k in k_values}
    true_labels_list = []
    probs_list = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluate @k'):
            logits = model(
                batch['input_ids'].to(device), batch['attention_mask'].to(device)
            )
            probabilities = torch.sigmoid(logits).cpu().numpy()
            labels = batch['labels'].cpu().numpy()

            for sample_probs, sample_labels in zip(probabilities, labels):
                true_indices = set(np.flatnonzero(sample_labels == 1))
                if not true_indices:
                    continue
                ranked_indices = np.argsort(sample_probs)[::-1]
                for k in k_values:
                    top_k = set(ranked_indices[:k])
                    tp = len(top_k.intersection(true_indices))
                    precision = tp / k
                    recall = tp / len(true_indices)
                    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
                    precisions[k].append(precision)
                    recalls[k].append(recall)
                    f1_scores[k].append(f1)
                true_labels_list.append(sample_labels)
                probs_list.append(sample_probs)

    results = {}
    for k in k_values:
        avg_p = float(np.mean(precisions[k])) if precisions[k] else 0.0
        avg_r = float(np.mean(recalls[k])) if recalls[k] else 0.0
        avg_f1 = float(np.mean(f1_scores[k])) if f1_scores[k] else 0.0
        results[k] = {
            'Precision': avg_p, 'Recall': avg_r, 'F1': avg_f1,
            'Repositories': len(precisions[k]),
        }
        print(f'--- K={k} ---')
        print(f'Precision@{k}: {avg_p * 100:.2f}%')
        print(f'Recall@{k}:    {avg_r * 100:.2f}%')
        print(f'F1@{k}:        {avg_f1 * 100:.2f}%')
        print(f'Repositories:  {len(precisions[k])}')
    return results, true_labels_list, probs_list

train_metrics_report, train_true_labels_list, train_probs_list = evaluate_at_k(
    bert_model, train_dataloader
)
test_metrics_report, test_true_labels_list, test_probs_list = evaluate_at_k(
    bert_model, test_dataloader
)

## Best probability threshold using samples F1

In [ ]:
def find_best_probability_threshold(
    true_labels_list, probs_list, thresholds=np.arange(0.01, 1.0, 0.01)
):
    y_true = np.asarray(true_labels_list, dtype=np.int8)
    y_prob = np.asarray(probs_list, dtype=np.float32)
    if y_true.shape != y_prob.shape:
        raise ValueError(f'Shape mismatch: labels={y_true.shape}, probabilities={y_prob.shape}')

    best_threshold = None
    best_f1 = -1.0
    for threshold in tqdm(thresholds, desc='Search threshold'):
        y_pred = (y_prob >= threshold).astype(np.int8)
        score = f1_score(y_true, y_pred, average='samples', zero_division=0)
        if score > best_f1:
            best_threshold = float(threshold)
            best_f1 = float(score)

    print(f'Best threshold: {best_threshold:.2f}')
    print(f'Best samples F1: {best_f1 * 100:.2f}%')
    return best_threshold, best_f1

best_threshold, best_f1 = find_best_probability_threshold(
    test_true_labels_list, test_probs_list
)

## Hit rate at the selected threshold

In [ ]:
def calculate_hit_rate_bert(
    model, dataloader, threshold, hit_targets=(1, 2, 3, 5, 10), device=device
):
    model.eval()
    success_counts = {target: 0 for target in hit_targets}
    valid_repos_counts = {target: 0 for target in hit_targets}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Hit rate'):
            logits = model(
                batch['input_ids'].to(device), batch['attention_mask'].to(device)
            )
            predicted_labels = torch.sigmoid(logits).cpu().numpy() >= threshold
            true_labels = batch['labels'].cpu().numpy()

            for y_true, y_pred in zip(true_labels, predicted_labels):
                true_indices = set(np.flatnonzero(y_true == 1))
                if not true_indices:
                    continue
                predicted_indices = set(np.flatnonzero(y_pred))
                tp = len(true_indices.intersection(predicted_indices))
                for target in hit_targets:
                    if len(true_indices) >= target:
                        valid_repos_counts[target] += 1
                        if tp >= target:
                            success_counts[target] += 1

    results = {}
    print('HIT RATE:')
    for target in hit_targets:
        valid_count = valid_repos_counts[target]
        success_count = success_counts[target]
        hit_rate = success_count / valid_count if valid_count else 0.0
        results[target] = {
            'success_count': success_count,
            'valid_repos_count': valid_count,
            'hit_rate': hit_rate,
        }
        print(
            f'Correctly guessed >= {target}: {success_count} / {valid_count} '
            f'repos ({hit_rate * 100:.1f}%)'
        )
    return results

test_hit_rates = calculate_hit_rate_bert(
    bert_model, test_dataloader, best_threshold
)

## Recommend packages for one description

In [ ]:
def recommend_packages_bert(
    model, description, tokenizer, mlb, threshold, max_length=MAX_LENGTH, device=device
):
    model.eval()
    encoding = tokenizer(
        description,
        truncation=True,
        max_length=max_length,
        padding=True,
        return_tensors='pt',
    )
    with torch.no_grad():
        logits = model(
            encoding['input_ids'].to(device),
            encoding['attention_mask'].to(device),
        )
        probabilities = torch.sigmoid(logits)[0].cpu().numpy()

    indices = np.flatnonzero(probabilities >= threshold)
    indices = indices[np.argsort(probabilities[indices])[::-1]]
    recommended = [mlb.classes_[index] for index in indices]
    scores = {mlb.classes_[index]: float(probabilities[index]) for index in indices}
    return recommended, scores

example_recommendations, example_scores = recommend_packages_bert(
    bert_model, test_df.iloc[0]['desc'], tokenizer, mlb, best_threshold
)
print('True:', test_df.iloc[0]['deps'])
print('Recommended:', example_recommendations)
print('Scores:', example_scores)

## Save model, tokenizer and label mapping

In [ ]:
OUTPUT_DIR = 'bert_package_recommender'
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.save(bert_model.state_dict(), os.path.join(OUTPUT_DIR, 'model.pt'))
tokenizer.save_pretrained(OUTPUT_DIR)
np.save(os.path.join(OUTPUT_DIR, 'classes.npy'), mlb.classes_)
np.save(os.path.join(OUTPUT_DIR, 'best_threshold.npy'), np.array(best_threshold))
print('Saved to:', OUTPUT_DIR)